# Project CV2: Robust Head Pose Estimation Under Facial Occlusions

Work by Antonio Vila Leis and Enric Baixauli Casañ

## Introduction

This project addresses the challenge of accurately estimating head posture in the presence of facial occlusions, such as masks and sunglasses.

TODO explicar todo

In [1]:
import os
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader
import sys
from dataset import LP300W, I2HeadDataset
import utils
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import time
from models.utils_hpe import compute_euler_angles_from_rotation_matrices
import cv2
import torch._dynamo
torch._dynamo.config.suppress_errors = True

C:\Users\enric\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\face_alignment\api.py:130: UserWarning: Compiling face alignment model (one-time cost). Subsequent runs will be faster.
  warnings.warn(
torch.compile failed (backend='inductor' raised:
RuntimeError: Cannot find a working triton installation. Either the package is not installed or it is too old. More information on installing Triton can be found at https://github.com/openai/triton

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True
), using eager mode


In [2]:
utils.set_global_seed(123)

if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
device = torch.device(device_name)
print(f"Code runs in {device}")

Code runs in cuda


## 1. Data Loading

In [3]:
import os
import random
from torch.utils.data import DataLoader
import torchvision.transforms as T

RAW_DATASET_DIR = "./dataset/test/raw"
BATCH_SIZE = 32

transforms_pipeline = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

subjects_dict = {}

# Parse baseline references from the raw directory
for file in os.listdir(RAW_DATASET_DIR):
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
        filename_no_ext = os.path.splitext(file)[0]
        
        # Extract subject prefix (e.g., pulls 'p01' from 'p01_image_01_raw' or 'p01_image_01')
        subject_id = filename_no_ext.split("_")[0] 
        
        full_path = os.path.join(RAW_DATASET_DIR, file)
        
        if subject_id not in subjects_dict:
            subjects_dict[subject_id] = []
        subjects_dict[subject_id].append(full_path)

unique_subjects = sorted(list(subjects_dict.keys()))
print(f"Total number of people found: {len(unique_subjects)}")

# Subject-level Split (10 Train / 1 Val / 1 Test)
random.seed(123)
random.shuffle(unique_subjects)

train_subj = unique_subjects[:9]
val_subj   = [unique_subjects[9]]
test_subj  = unique_subjects[10:]

print(f"Split people -> Train: {train_subj} | Val: {val_subj} | Test: {test_subj}")

# Gather lists of paths
train_paths = []
for subj in train_subj:
    train_paths.extend(subjects_dict[subj])

val_paths = []
for subj in val_subj:
    val_paths.extend(subjects_dict[subj])

test_paths = []
for subj in test_subj:
    test_paths.extend(subjects_dict[subj])

print(f"Split images -> Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

# Instantiate Datasets
train_dataset = I2HeadDataset(image_paths_or_dir=train_paths, transform=transforms_pipeline, occlusion_mode="random")
val_dataset   = I2HeadDataset(image_paths_or_dir=val_paths, transform=transforms_pipeline, occlusion_mode="random")

# Define the 4 isolated evaluation tracks matching the test subject
test_raw     = I2HeadDataset(image_paths_or_dir=test_paths, transform=transforms_pipeline, occlusion_mode="raw")
test_mask    = I2HeadDataset(image_paths_or_dir=test_paths, transform=transforms_pipeline, occlusion_mode="mask")
test_glasses = I2HeadDataset(image_paths_or_dir=test_paths, transform=transforms_pipeline, occlusion_mode="glasses")
test_both    = I2HeadDataset(image_paths_or_dir=test_paths, transform=transforms_pipeline, occlusion_mode="both")

# Construct DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

raw_loader     = DataLoader(test_raw, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
mask_loader    = DataLoader(test_mask, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
glasses_loader = DataLoader(test_glasses, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
both_loader    = DataLoader(test_both, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print("DataLoaders built successfully!")

Total number of people found: 12
Split people -> Train: ['p08', 'p06', 'p10', 'p03', 'p04', 'p09', 'p12', 'p11', 'p07'] | Val: ['p02'] | Test: ['p05', 'p01']
Split images -> Train: 738 | Val: 82 | Test: 164
DataLoaders built successfully!


In [4]:
sys.path.append("./models")

from token_hpe import TokenHPE

model = TokenHPE(depth=3)
checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-lyr3.tar", map_location="cpu")

state_dict = checkpoint["model_state_dict"]

model.load_state_dict(state_dict, strict=True)

model = model.to(device)

==> Add Sine PositionEmbedding~


C:\Users\enric\AppData\Local\Temp\ipykernel_22956\1715322118.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-l

In [5]:
model.eval()

print("\n-> Test 1/4: RAW")
mae_r, rmse_r, r2_r = utils.evaluate_metrics(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
mae_m, rmse_m, r2_m = utils.evaluate_metrics(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
mae_g, rmse_g, r2_g = utils.evaluate_metrics(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
mae_b, rmse_b, r2_b = utils.evaluate_metrics(model, both_loader, device)

def print_results(test_name, mae, rmse, r2):
    print(f"\n{test_name}")
    print(f"  MAE -> Yaw: {mae['yaw']} | Pitch: {mae['pitch']} | Roll: {mae['roll']} | Total: {mae['total']}")
    print(f"  RMSE -> Yaw: {rmse['yaw']} | Pitch: {rmse['pitch']} | Roll: {rmse['roll']} | Total: {rmse['total']}")
    print(f"  R² -> Yaw: {r2['yaw']} | Pitch: {r2['pitch']} | Roll: {r2['roll']} | Total: {r2['total']}")

print_results("Test RAW:", mae_r, rmse_r, r2_r)
print_results("Test MASK:", mae_m, rmse_m, r2_m)
print_results("Test GLASSES:", mae_g, rmse_g, r2_g)
print_results("Test BOTH:", mae_b, rmse_b, r2_b)


-> Test 1/4: RAW


Evaluating: 100%|██████████| 6/6 [00:32<00:00,  5.39s/it]



-> Test 2/4: MASK


Evaluating: 100%|██████████| 6/6 [00:30<00:00,  5.15s/it]



-> Test 3/4: GLASSES


Evaluating: 100%|██████████| 6/6 [00:30<00:00,  5.02s/it]



-> Test 4/4: BOTH


Evaluating: 100%|██████████| 6/6 [00:30<00:00,  5.05s/it]


Test RAW:
  MAE -> Yaw: 0.12259089201688766 | Pitch: 0.1570242941379547 | Roll: 0.13681243360042572 | Total: 0.1388092041015625
  RMSE -> Yaw: 0.15527768433094025 | Pitch: 0.20764848589897156 | Roll: 0.1565522700548172 | Total: 0.17315948009490967
  R² -> Yaw: -0.22106695175170898 | Pitch: -46.26097106933594 | Roll: -1.1053576469421387 | Total: -15.862464904785156

Test MASK:
  MAE -> Yaw: 0.3934604227542877 | Pitch: 0.15674462914466858 | Roll: 0.08211459219455719 | Total: 0.21077321469783783
  RMSE -> Yaw: 0.4482227563858032 | Pitch: 0.20482006669044495 | Roll: 0.10380639135837555 | Total: 0.2522830665111542
  R² -> Yaw: -9.174407958984375 | Pitch: -44.982234954833984 | Roll: 0.07433187961578369 | Total: -18.027437210083008

Test GLASSES:
  MAE -> Yaw: 0.1747351437807083 | Pitch: 0.14715459942817688 | Roll: 0.1406664103269577 | Total: 0.15418539941310883
  RMSE -> Yaw: 0.20160287618637085 | Pitch: 0.1872008889913559 | Roll: 0.16447339951992035 | Total: 0.1844257265329361
  R² -> Yaw:

# Fine-Tunning

## Total Fine-Tunning

In [7]:
criterion = nn.L1Loss()

optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)

epochs = 5
best_val_loss = float('inf')

C:\Users\enric\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [8]:
train_losses = []
val_losses = []
def train_model(epochs=epochs, model=model, name = "base", train_loader=train_loader, val_loader=val_loader, criterion=criterion, optimizer=optimizer, scheduler=scheduler, device=device):
    global best_val_loss
    for epoch in range(epochs):
        start_time = time.time()

        # Training
        model.train()
        running_train_loss = 0.0
    
        for imgs, poses in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            imgs, poses = imgs.to(device), poses.to(device)
            
            optimizer.zero_grad()
            
            predictions = model(imgs)
            
            if isinstance(predictions, (tuple, list)):
                pred_mat = predictions[0]
                pred_angles = compute_euler_angles_from_rotation_matrices(pred_mat, use_gpu=True)
                pred_angles = pred_angles.to(device)
            else:
                pred_angles = predictions
                
            loss = criterion(pred_angles, poses)
            
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item() * imgs.size(0)
            
        epoch_train_loss = running_train_loss / len(train_dataset)
        train_losses.append(epoch_train_loss)
        
        # Validation
        model.eval()
        running_val_loss = 0.0
        
        with torch.no_grad():
            for imgs, poses in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                imgs, poses = imgs.to(device), poses.to(device)
                
                predictions = model(imgs)
                
                if isinstance(predictions, (tuple, list)):
                    pred_mat = predictions[0]
                    pred_angles = compute_euler_angles_from_rotation_matrices(pred_mat, use_gpu=True)
                    pred_angles = pred_angles.to(device)
                else:
                    pred_angles = predictions
                    
                loss = criterion(pred_angles, poses)
                running_val_loss += loss.item() * imgs.size(0)
                
        epoch_val_loss = running_val_loss / len(val_dataset)
        val_losses.append(epoch_val_loss)
        
        scheduler.step(epoch_val_loss)
        
        epoch_time = time.time() - start_time
        
        print(f"\nEpoch {epoch+1}/{epochs} completed in {epoch_time:.0f}s")
        print(f"Train Loss (MAE radians): {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            save_path = "./weights/TokenHPE_Finetuned_" + name + ".pth"
            torch.save(model.state_dict(), save_path)
            print("Best model updated")
        else:
            print("\n")

In [ ]:
train_model(name="Complete", epochs=epochs, model=model, optimizer=optimizer, scheduler=scheduler)

Epoch 1/5 [Train]:   0%|          | 0/1725 [00:00<?, ?it/s]

In [5]:
sys.path.append("./models")

from token_hpe import TokenHPE

model = TokenHPE(depth=3)
checkpoint_path = "./weights/TokenHPE_Finetuned_Best.pth"
checkpoint = torch.load(checkpoint_path, map_location="cpu")

state_dict = checkpoint.get("model_state_dict", checkpoint)
model.load_state_dict(state_dict, strict=True)

model = model.to(device)
model.eval()

print("\n-> Test 1/4: RAW")
yaw_r, pitch_r, roll_r, total_r = utils.evaluate_mae(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
yaw_m, pitch_m, roll_m, total_m = utils.evaluate_mae(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
yaw_g, pitch_g, roll_g, total_g = utils.evaluate_mae(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
yaw_b, pitch_b, roll_b, total_b = utils.evaluate_mae(model, both_loader, device)

print("Test results:")
print(f"Test RAW     -> Yaw: {yaw_r} | Pitch: {pitch_r} | Roll: {roll_r} | Total: {total_r}")
print(f"Test MASK    -> Yaw: {yaw_m} | Pitch: {pitch_m} | Roll: {roll_m} | Total: {total_m}")
print(f"Test GLASSES -> Yaw: {yaw_g} | Pitch: {pitch_g} | Roll: {roll_g} | Total: {total_g}")
print(f"Test BOTH    -> Yaw: {yaw_b} | Pitch: {pitch_b} | Roll: {roll_b} | Total: {total_b}")


==> Add Sine PositionEmbedding~


C:\Users\enric\AppData\Local\Temp\ipykernel_21376\1973561996.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cpu")

FileNotFoundError: [Errno 2] No such file or directory: './weights/TokenHPE_Finetuned_Best.pth'

## LoRA

In [ ]:
for name, m in model.feature_extractor.named_modules():
    if 'qkv' in name or 'attn' in name or 'proj' in name:
        print(name)

NameError: name 'model' is not defined

In [7]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["qkv"], # Targets the self-attention projections in the ViT blocks
    lora_dropout=0.1,
    bias="none"
)

model.feature_extractor = get_peft_model(model.feature_extractor, config)
model = model.to(device)

In [8]:
# We completely freeze all the parameters of the base model
for param in model.parameters():
    param.requires_grad = False

# We only unfreeze the parameters that contain "lora" and the downstream task modules
for name, param in model.named_parameters():
    if "lora" in name or "Ori_blocks" in name or "mlp_head" in name:
        param.requires_grad = True

# Here we can see how many parameters will actually be trained
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} out of {total_params:,} "
      f"({trainable_params/total_params:.2%})")

Trainable Parameters: 1,231,636 out of 87,034,906 (1.42%)


In [ ]:
criterion = nn.L1Loss()

optimizer_lora = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_lora, mode='min', patience=3, factor=0.5, verbose=True
)

best_val_loss = float('inf')
epochs = 10
best_val_loss = float('inf')
train_model(epochs=epochs, model=model, optimizer=optimizer_lora, scheduler=scheduler, name="LoRA")

Epoch 1/10 [Train]:   0%|          | 0/1531 [00:00<?, ?it/s]

In [ ]:
print("\n-> Test 1/4: RAW")
yaw_r, pitch_r, roll_r, total_r = utils.evaluate_mae(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
yaw_m, pitch_m, roll_m, total_m = utils.evaluate_mae(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
yaw_g, pitch_g, roll_g, total_g = utils.evaluate_mae(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
yaw_b, pitch_b, roll_b, total_b = utils.evaluate_mae(model, both_loader, device)

print("Test results:")
print(f"Test RAW     -> Yaw: {yaw_r} | Pitch: {pitch_r} | Roll: {roll_r} | Total: {total_r}")
print(f"Test MASK    -> Yaw: {yaw_m} | Pitch: {pitch_m} | Roll: {roll_m} | Total: {total_m}")
print(f"Test GLASSES -> Yaw: {yaw_g} | Pitch: {pitch_g} | Roll: {roll_g} | Total: {total_g}")
print(f"Test BOTH    -> Yaw: {yaw_b} | Pitch: {pitch_b} | Roll: {roll_b} | Total: {total_b}")


-> Test 1/4: RAW



-> Test 1/4: RAW


Evaluating: 100%|██████████| 192/192 [00:56<00:00,  3.42it/s]



-> Test 1/4: RAW


Evaluating: 100%|██████████| 192/192 [00:56<00:00,  3.42it/s]


...


-> Test 3/4: GLASSES


Evaluating: 100%|██████████| 192/192 [00:53<00:00,  3.60it/s]



-> Test 4/4: BOTH


Evaluating: 100%|██████████| 192/192 [00:53<00:00,  3.62it/s]

Test results:
Test RAW     -> Yaw: 0.03269583437541293 | Pitch: 0.04455014950365057 | Roll: 0.03719413530664975 | Total: 0.038146706395237755
Test MASK    -> Yaw: 0.041198861982509744 | Pitch: 0.04957511935887453 | Roll: 0.03904164884870356 | Total: 0.043271876730029274
Test GLASSES -> Yaw: 0.033368440151759736 | Pitch: 0.04786138518581238 | Roll: 0.039913227993075645 | Total: 0.040381017776882584
Test BOTH    -> Yaw: 0.03919568723583268 | Pitch: 0.047669987711235914 | Roll: 0.0385098766408363 | Total: 0.04179185052930163


# TODO list

## Probar con oclusion sencilla

## Buscar como hacer un test con imagenes reales

## Cambiar métricas a RMSE y R² (o MAPE con comprobacion de que no divida entre 0)



In [2]:
from datasets import load_dataset

# Carregar el dataset (descarrega automàticament les imatges i anotacions)
dataset = load_dataset("ETHZurich/biwi_kinect_head_pose", trust_remote_code=True)

# Exemple d'estructura d'una mostra:
sample = dataset['train'][0]
print(sample.keys())
# Retorna dict_keys(['sequence_number', 'subject_id', 'rgb', 'rgb_cal', 'depth'])

FileNotFoundError: https://data.vision.ee.ethz.ch/cvl/gfanelli/kinect_head_pose_db.tgz

In [8]:
import scipy.io
import torch

mat_data = scipy.io.loadmat('p02_image_04.mat')
vector_6d = mat_data['HP_camera'][0]  

print("keys:", mat_data.keys())

angle_1 = vector_6d[3]
angle_2 = vector_6d[4]
angle_3 = vector_6d[5]

print(f"Angle 1: {angle_1}, Angle 2: {angle_2}, Angle 3: {angle_3}")

radians = torch.tensor([angle_1, angle_2, angle_3]) * (torch.pi / 180)
print(radians)

keys: dict_keys(['__header__', '__version__', '__globals__', 'HP_camera'])
Angle 1: 7.0333972381660965, Angle 2: 23.097407019507475, Angle 3: 12.855895204825982
tensor([0.1228, 0.4031, 0.2244], dtype=torch.float64)


In [9]:
import scipy.io as sio


hp = mat_data["HP_camera"]
print(hp.shape)
print(hp[0])


(1, 6)
[  -7.91547312 -172.01305958  638.96059032    7.03339724   23.09740702
   12.8558952 ]
